# Lab 3 demo: Track B retention-offer RAG pipeline

End-to-end run of the pipeline in `rag/`: policy corpus -> section chunks -> Titan V2 embeddings in FAISS -> two-pass retrieval -> Claude Haiku 4.5 offer, for the four `OFFER_TEST_CASES` in `evaluation_harness.py`. The RAGAS scores shown at the end are loaded from the committed evidence files (`docs/lab3-evidence/rag_eval.csv`, `rag_eval_strategies.csv`), so this notebook makes only 5 generator calls plus the query embeddings (about 2 cents).

Run with **Kernel > Restart & Run All** from the repo's `.venv` with AWS credentials for `us-east-1`.

In [1]:
import os, sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore", message="IProgress not found")   # tqdm widget notice, no effect

# Work from the repo root whether the kernel started in notebooks/ or the root.
root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(root); sys.path.insert(0, str(root))
import pandas as pd
pd.set_option("display.max_colwidth", 120); pd.set_option("display.width", 160)
print("repo root:", root)

repo root: D:\dev\bost\cs401r-bsharp02


## 1. Corpus and index

In [2]:
from rag.corpus import load_documents, chunk_by_section
from rag.index import PolicyIndex

docs = load_documents()
chunks = chunk_by_section(docs)
sizes = [len(c.text.split()) for c in chunks]
print(f"{len(docs)} policy documents -> {len(chunks)} section chunks; "
      f"words/chunk min {min(sizes)} median {sorted(sizes)[len(sizes)//2]} max {max(sizes)}")
index = PolicyIndex.build("section")        # cached under rag/index/section/, no re-embedding
print("index vectors:", index.faiss.ntotal, "x", index.faiss.d)
example = next(c for c in chunks if c.chunk_id == "POL-LOY-011#4")
print(f"\n[{example.chunk_id}] {example.text[:400]}...")

4 policy documents -> 48 section chunks; words/chunk min 23 median 71 max 252
index vectors: 48 x 1024

[POL-LOY-011#4] NorthStar Summit Rewards — Program Terms (POL-LOY-011) — 4. Tier Benefits
| Benefit | Bronze | Silver | Gold | Platinum |
|---|---|---|---|---|
| Points earn rate | 1× | 1.25× | 1.5× | 2× |
| Birthday reward | $5 | $10 | $20 | $40 |
| Free standard shipping | $75+ | $50+ | Always | Always |
| Free return shipping | No | No | Yes | Yes |
| Return window | 30 days | 30 days | 30 days | **60 days** |...


## 2. The four test cases through the pipeline

In [3]:
from evaluation_harness import OFFER_TEST_CASES
from rag.pipeline import OfferPipeline

pipe = OfferPipeline(strategy="AB", chunking="section", k=5, profile_in_contexts=True)
print("config:", json.dumps(pipe.config))
results = []
for tc in OFFER_TEST_CASES:
    ctx = dict(tc.customer_context, customer_id=tc.customer_id)
    r = pipe.invoke(ctx, tc.question)
    results.append(r)
    print(f"\n=== {tc.customer_id} ({ctx['loyalty_tier']}, {ctx['days_since_last_purchase']} days lapsed, "
          f"{', '.join(ctx['top_categories'])})")
    print("retrieved:", ", ".join(r["chunk_ids"]))
    print(r["answer"])
    print(f"[{r['input_tokens']} in / {r['output_tokens']} out tokens, {r['latency_ms']} ms]")

config: {"retrieval": "AB", "chunking": "section", "k": 5, "profile_in_contexts": true, "temperature": 0.2}



=== CUST-10000317 (Gold, 47 days lapsed, Outdoor, Footwear)
retrieved: POL-LOY-011#4, KB-FAQ-022#loyalty, POL-LOY-011#2.3, POL-LOY-011#2.1, POL-RET-004#1.1, POL-LOY-011#5.1, POL-LOY-011#5, POL-RET-004#4, KB-FAQ-022#returns, KB-FAQ-022#guidance-for-automated-agents
As a Gold member, you currently receive free return shipping on all orders, which applies to your outdoor gear and footwear purchases. Your points earn at 1.5× the standard rate, and any order qualifies for free standard shipping—no minimum spend required. Since your last purchase 47 days ago, redeeming 100 of your accumulated points for $5 toward your next order will reset your points expiration clock to 18 months from that purchase date. This offer is valid for 45 days.
[4031 in / 113 out tokens, 1803.9 ms]



=== CUST-10000904 (Bronze, 68 days lapsed, Apparel)
retrieved: POL-LOY-011#4, KB-FAQ-022#loyalty, POL-RET-004#1.1, POL-LOY-011#2.3, POL-LOY-011#2, POL-LOY-011#5.1, POL-LOY-011#5, POL-RET-004#4, KB-FAQ-022#returns, KB-FAQ-022#guidance-for-automated-agents
As a Bronze member, you earn 1× points on every apparel purchase and can redeem 100 points for $5 in store credit. Your points balance stays active for 18 months after your most recent purchase, so returning to apparel now will reset that clock and keep your rewards within reach. Stop by any NorthStar location to browse new apparel and apply your points toward your next transaction. This offer is valid for 45 days from today.
[4166 in / 102 out tokens, 1916.9 ms]



=== CUST-10000122 (Platinum, 39 days lapsed, Outdoor, Electronics, Home)
retrieved: POL-LOY-011#4, KB-FAQ-022#loyalty, POL-RET-004#1.1, POL-LOY-011#2.3, POL-LOY-011#2, POL-LOY-011#5.1, POL-LOY-011#5, POL-RET-004#4, KB-FAQ-022#returns, KB-FAQ-022#guidance-for-automated-agents
As a Platinum member, you earn 2× points on every purchase across Outdoor, Electronics, and Home categories, and your points don't expire as long as you make a purchase within 18 months—your clock resets with each order. Your $100 annual gear credit is available now to apply toward any eligible merchandise, and you have 60 days to return items if they don't work out. You also receive free return shipping on all orders and 48-hour early access to upcoming sales. This offer is valid for 45 days.
[4136 in / 119 out tokens, 2077.4 ms]



=== CUST-10000588 (Silver, 52 days lapsed, Home, Beauty)
retrieved: POL-LOY-011#4, KB-FAQ-022#loyalty, POL-LOY-011#2, POL-LOY-011#2.3, POL-RET-004#1.1, POL-LOY-011#5.1, POL-LOY-011#5, POL-RET-004#4, KB-FAQ-022#returns, KB-FAQ-022#guidance-for-automated-agents
As a Silver member, you earn 1.25× points on every purchase, and your points don't expire as long as you make a purchase within 18 months—your clock resets with each order. Your last purchase was 52 days ago, so you're well within that window and can start fresh with a new buy in Home or Beauty. Free standard shipping kicks in at $50 for Silver members, and you're just $15 away from Gold status at $2,000 lifetime spend; a purchase in either category moves you closer to the 24-hour early sale access and free return shipping that Gold members receive. This offer is valid for 45 days.
[4133 in / 146 out tokens, 2470.1 ms]


## 3. What the before/after flag changes

`profile_in_contexts` does not change the generator prompt at all. It changes what the judge is allowed to count as support: with the flag on, the customer profile is appended as one more context, so profile facts in the offer (tier, categories, days since last purchase) can be verified instead of being marked unsupported.

In [4]:
tc = OFFER_TEST_CASES[0]
ctx = dict(tc.customer_context, customer_id=tc.customer_id)
before = OfferPipeline("AB", "section", 5, profile_in_contexts=False)
r_before = before.invoke(ctx, tc.question)   # one more generator call (sub-cent)
r_after = results[0]
print("contexts the judge sees, before:", len(r_before["contexts"]), "| after:", len(r_after["contexts"]))
print("\nthe extra 'after' context:\n" + r_after["contexts"][-1])

contexts the judge sees, before: 10 | after: 11

the extra 'after' context:
Customer profile:
- churn_probability: 81%
- loyalty_tier: Gold
- days_since_last_purchase: 47
- total_lifetime_value: $3,120.40
- avg_order_value: $186.25
- category_diversity_score: 0.5
- online_to_store_ratio: 0.72
- top_categories: Outdoor, Footwear
- customer_id: CUST-10000317


## 4. RAGAS scores from the evidence files

Judge: `us.anthropic.claude-haiku-4-5-20251001-v1:0` (temperature 0, max_tokens 4096) through RAGAS 0.4.3. Two runs per label, four cases each. `before` = policy chunks only, `after` = policy chunks + profile.

In [5]:
ev = pd.read_csv("docs/lab3-evidence/rag_eval.csv")
metrics = ["faithfulness", "answer_relevancy", "context_recall"]
per_run = ev.groupby(["label", "run"])[metrics].mean().round(3)
summary = per_run.groupby("label").agg(["mean", "min", "max"]).round(3)
display(per_run)
display(summary)

faithfulness  answer_relevancy  context_recall
label  run                                                
after  1           0.553             0.386           0.733
       2           0.556             0.395           0.608
before 1           0.383             0.350           0.671
       2           0.332             0.396           0.671

faithfulness               answer_relevancy               context_recall              
               mean    min    max             mean    min    max           mean    min    max
label                                                                                        
after         0.554  0.553  0.556            0.390  0.386  0.395          0.670  0.608  0.733
before        0.358  0.332  0.383            0.373  0.350  0.396          0.671  0.671  0.671

In [6]:
per_case = ev.pivot_table(index=["customer_id", "loyalty_tier"], columns=["label", "run"], values=metrics).round(3)
display(per_case)

answer_relevancy                      context_recall                      faithfulness                     
label                                 after        before                 after        before               after        before       
run                                       1      2      1      2              1      2      1      2            1      2      1      2
customer_id   loyalty_tier                                                                                                            
CUST-10000122 Platinum                0.324  0.343  0.353  0.353          0.333  0.333  0.333  0.333        0.538  0.500  0.500  0.500
CUST-10000317 Gold                    0.408  0.408  0.400  0.408          1.000  1.000  0.750  0.750        0.444  0.333  0.385  0.222
CUST-10000588 Silver                  0.343  0.359  0.343  0.352          1.000  0.500  1.000  1.000        0.727  0.889  0.364  0.308
CUST-10000904 Bronze                  0.471  0.470  0.303  0.471          0.600  0.600  0.600  0.600        0.500  0.500  0.286  0.300

## 5. Retrieval strategy comparison (Template 4)

One run per strategy with the profile in contexts. `A` = tier-benefit query, `B` = constraint query, `C` = hybrid query restricted to the loyalty and shipping documents, `AB` = A then B merged (the default).

In [7]:
st = pd.read_csv("docs/lab3-evidence/rag_eval_strategies.csv")
display(st.groupby("label")[metrics].mean().round(3).loc[["strategy-A", "strategy-B", "strategy-C", "strategy-AB"]])
display(st.pivot_table(index="customer_id", columns="label", values="context_recall").round(3))
print("chunks strategy B retrieved for every customer:")
print(st[st.label == "strategy-B"].chunk_ids.unique())

,faithfulness,answer_relevancy,context_recall
label,,,
strategy-A,0.482,0.347,0.583
strategy-B,0.281,0.360,0.350
strategy-C,0.556,0.339,0.508
strategy-AB,0.553,0.386,0.733


label,strategy-A,strategy-AB,strategy-B,strategy-C
customer_id,,,,
CUST-10000122,0.333,0.333,0.00,0.333
CUST-10000317,1.000,1.000,0.75,1.000
CUST-10000588,0.600,1.000,0.25,0.500
CUST-10000904,0.400,0.600,0.40,0.200


chunks strategy B retrieved for every customer:
<ArrowStringArray>
['POL-LOY-011#5.1;POL-LOY-011#5;POL-RET-004#4;KB-FAQ-022#returns;KB-FAQ-022#guidance-for-automated-agents']
Length: 1, dtype: str


## 6. Reading

- Adding the profile to the judge's contexts moves faithfulness from about 0.36 to about 0.55; it is the single pipeline change reported in `docs/lab3-evaluation-report.md`.
- Context recall is capped by the ground truths, not by retrieval: the Platinum case scores 0.333 under every strategy because two of its three ground-truth sentences are grading commentary that no policy chunk states.
- Strategy B on its own retrieves the same five retention-rule chunks for every customer and never sees the tier-benefit table, which is why its faithfulness and recall collapse; the two-pass merge exists to cover both needs.